# 01 — Naive RAG Architecture

O ponto de partida para entender arquiteturas RAG.

## Diagrama

```
INDEXING:
  Documentos
      ↓ Chunking
  Chunks de texto
      ↓ Embedding Model
  Vetores
      ↓ Store
  Vector Database

QUERYING:
  Query do usuario
      ↓ Embedding Model (mesmo!)
  Query Vector
      ↓ ANN Search (top-k)
  Chunks relevantes
      ↓ Prompt assembly
  [Contexto + Query] → LLM → Resposta
```

## Quando usar Naive RAG
- Prototipagem rapida
- Documentos bem estruturados e queries claras
- Quando latencia e prioridade

## Limitacoes
- Sem otimizacao de query
- Sem re-ranking de resultados
- Resultado depende muito da qualidade do chunking

In [ ]:
import sys
sys.path.insert(0, '..')

from src.rag.naive import NaiveRAG
from pathlib import Path

# Usar o modulo pronto de src/
rag = NaiveRAG(
    collection_name='arch_naive',
    embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    llm_model='llama3.2',
    chunk_size=512,
    chunk_overlap=64,
    top_k=5,
)

print(repr(rag))

In [ ]:
# Indexar documentos de exemplo
docs_dir = Path('../data/sample_docs')
documentos = [
    {'text': p.read_text(encoding='utf-8'), 'fonte': p.name}
    for p in docs_dir.glob('*.md')
]

n_chunks = rag.index(documentos, recreate=True)
print(f'Total de chunks indexados: {n_chunks}')

In [ ]:
# Benchmark de latencia
import time

perguntas = [
    'O que e RAG e como funciona?',
    'Quais sao os tipos de quantizacao vetorial?',
    'Como funciona o mecanismo de atencao nos Transformers?',
]

for p in perguntas:
    t0 = time.time()
    resultado = rag.query(p, verbose=False)
    latencia = time.time() - t0
    
    print(f'\nQ: {p}')
    print(f'A: {resultado["answer"][:200]}...')
    print(f'Latencia: {latencia:.2f}s | {resultado["num_sources"]} chunks')

## Analise de Falhas do Naive RAG

O Naive RAG falha em 4 cenarios principais:

In [ ]:
# Cenarios de falha
casos_de_falha = [
    {
        'descricao': 'Query ambigua (sem rewriting)',
        'query': 'Como e que funciona?',  # muito vaga
        'problema': 'Sem contexto de dominio, recupera chunks genericos',
    },
    {
        'descricao': 'Pergunta multi-step',
        'query': 'Qual modelo de embedding e mais adequado para um RAG de producao com restricao de memoria?',
        'problema': 'Precisa combinar infos de embeddings + quantizacao + memoria — um chunk raramente tem tudo',
    },
    {
        'descricao': 'Query com negacao',
        'query': 'Qual metrica NAO deve ser usada para texto sem normalizacao?',
        'problema': 'Embedding de queries com negacao pode recuperar o oposto do esperado',
    },
]

for caso in casos_de_falha:
    print(f'\n[{caso["descricao"]}]')
    print(f'Query: {caso["query"]}')
    
    # Mostrar apenas o retrieval
    chunks = rag.retrieve(caso['query'])
    print(f'Top chunk recuperado (score={chunks[0]["score"]:.3f}):')
    print(f'  {chunks[0]["payload"].get("text", "")[:100]}...')
    print(f'Problema: {caso["problema"]}')

## Metricas do Naive RAG

| Metrica | Valor tipico | Observacao |
|---------|-------------|------------|
| Latencia total | 2-5s | Embedding + search + LLM |
| Latencia sem LLM | 50-200ms | Embedding + Qdrant search |
| Hit@1 (retrieval) | 60-75% | Depende do chunking |
| Faithfulness | 70-85% | Depende do prompt |
| Answer Relevancy | 75-85% | Depende da qualidade do retrieval |

## Como melhorar → Advanced RAG

```
Problema: Query ruim → Solucao: Query rewriting
Problema: Chunks errados no top-k → Solucao: Re-ranking
Problema: Contexto muito longo → Solucao: Context compression
Problema: Pergunta multi-hop → Solucao: Agentic RAG
```

## Proximo
- [02 — Advanced RAG](02_advanced_rag.ipynb)